### Step 3: Recommendation System using Cosine Similarity

In [21]:
import warnings
warnings.filterwarnings("ignore")

####  3.1 IMPORTANT RULE This creates a huge matrix → Memory Error and Compute similarity only when needed

3.2 Load Data

In [15]:
import pandas as pd
import pickle

mlb = pickle.load(open('mlb.pkl', 'rb'))
scaler = pickle.load(open('scaler.pkl', 'rb'))
encoded_df = pd.read_csv('encoded_data.csv')
cleaned_df = pd.read_csv('cleaned_data.csv')

####  3.3 Recommendation by Index

In [59]:
from sklearn.metrics.pairwise import cosine_similarity


def recommend_restaurants(index, top_n=5):

    # Compute similarity for only one row
    similarity_scores = cosine_similarity(
        encoded_df.iloc[index:index+1],
        encoded_df
    )

    # Convert to list
    scores = list(enumerate(similarity_scores[0]))

    # Sort (highest similarity first)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Remove itself
    scores = scores[1:top_n+1]

    # Get indices
    indices = [i[0] for i in scores]

    # ✅ Add maincity in output
    return cleaned_df.iloc[indices][
        ['name', 'maincity', 'city', 'cuisine', 'rating', 'cost']
    ]

In [60]:
recommend_restaurants(10)

,name,maincity,city,cuisine,rating,cost
42,The Super Cafe,Abohar,Abohar,Beverages,3.80,250
49,FresHub,Abohar,Abohar,Beverages,3.80,200
2,theka coffee desi,Abohar,Abohar,Beverages,3.80,100
38289,Juice Box,Chennai,Tambaram,"Juices,Beverages",4.05,250
39949,THE JUICE FACTORY,Chennai,Porur,"Beverages,Juices",4.05,250


3.4 User-Based Recommendation 

In [61]:
def create_user_vector(user_input):

    # Default values
    default_input = {
        'maincity': None,
        'city': None,
        'cuisine': None,
        'cost': 0,
        'rating': 0
    }

    default_input.update(user_input)

    user_df = pd.DataFrame([default_input])

    # ✅ Encode maincity
    if default_input['maincity'] is not None:
        maincity_encoded = pd.get_dummies(
            user_df['maincity'], prefix='maincity')
    else:
        maincity_encoded = pd.DataFrame()

    # ✅ Encode city
    if default_input['city'] is not None:
        city_encoded = pd.get_dummies(user_df['city'], prefix='city')
    else:
        city_encoded = pd.DataFrame()

    # ✅ Encode cuisine
    if default_input['cuisine'] is not None:
        cuisine_list = default_input['cuisine'].split(',')
        cuisine_encoded = mlb.transform([cuisine_list])
        cuisine_df = pd.DataFrame(cuisine_encoded, columns=mlb.classes_)
    else:
        cuisine_df = pd.DataFrame(columns=mlb.classes_)

    # ✅ Combine everything
    final_user = pd.concat([
        user_df[['cost', 'rating']],
        maincity_encoded,
        city_encoded,
        cuisine_df
    ], axis=1)

    # ✅ Match with training columns
    final_user = final_user.reindex(columns=encoded_df.columns, fill_value=0)

    # ✅ Scale numeric
    final_user[['cost', 'rating']] = scaler.transform(
        final_user[['cost', 'rating']]
    )

    return final_user

In [111]:
from sklearn.metrics.pairwise import cosine_similarity


def recommend_for_user(user_vector, user_input, top_n=5):

    df_filtered = cleaned_df.copy()
    encoded_filtered = encoded_df.copy()

    # ✅ MAINCITY filter
    if user_input.get('maincity'):
        mask = df_filtered['maincity'].str.lower(
        ) == user_input['maincity'].lower()
        df_filtered = df_filtered[mask]
        encoded_filtered = encoded_filtered[mask]

    # ✅ CITY filter
    if user_input.get('city'):
        mask = df_filtered['city'].str.lower() == user_input['city'].lower()
        df_filtered = df_filtered[mask]
        encoded_filtered = encoded_filtered[mask]

    # ✅ CUISINE filter (FIXED ✅)
    if user_input.get('cuisine'):
        cuisine_list = [c.strip().lower()
                        for c in user_input['cuisine'].split(',')]

        mask = df_filtered['cuisine'].str.lower().apply(
            lambda x: any(c in x for c in cuisine_list)
        )

        df_filtered = df_filtered[mask]
        encoded_filtered = encoded_filtered[mask]

    # ✅ Reset index
    df_filtered = df_filtered.reset_index(drop=True)
    encoded_filtered = encoded_filtered.reset_index(drop=True)

    # ✅ Fallback
    if len(df_filtered) == 0:
        df_filtered = cleaned_df.reset_index(drop=True)
        encoded_filtered = encoded_df.reset_index(drop=True)

    # ✅ Similarity
    similarity = cosine_similarity(user_vector, encoded_filtered)

    scores = list(enumerate(similarity[0]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    top_indices = [i[0] for i in scores[:top_n]]

    return df_filtered.iloc[top_indices][
        ['name', 'maincity', 'city', 'cuisine', 'rating', 'cost']
    ]

In [116]:
user_input = {
    'maincity': 'Chennai',
    'city': 'Tambaram',
    'cuisine': 'Fast Food',
    'cost': 100,
    'rating':5
}

user_vec = create_user_vector(user_input)

recommend_for_user(user_vec, user_input)

,name,maincity,city,cuisine,rating,cost
25,TN 72 CAFE,Chennai,Tambaram,"Fast Food,Chinese",4.3,150
17,Le Wrap Factorie,Chennai,Tambaram,"Fast Food,Chinese",4.3,250
39,HK foods,Chennai,Tambaram,"Fast Food,Chinese",4.3,250
28,We Chai- Chitlapakkam,Chennai,Tambaram,"Beverages,Fast Food",4.2,102
1,Juicy Day,Chennai,Tambaram,"Beverages,Fast Food",4.2,200


In [ ]:
def evaluate_model(sample_size=5):
    correct = 0
    total = 0

    for i in range(sample_size):
        recs = recommend_restaurants(i, top_n=5)

        original = cleaned_df.iloc[i]

        for _, row in recs.iterrows():
            # Check same city OR same cuisine
            if (row['city'] == original['city']) or \
               (original['cuisine'] in row['cuisine']):
                correct += 1

            total += 1

    print("Accuracy:", correct / total)

In [ ]:
evaluate_model()

In [121]:
avg_similarity()

Average similarity: 0.03685379634855367
